# TCN-MLP Hybrid Model for Crop Yield Prediction
## Best Configuration: L2=1e-3, Dropout=0.25, n_aug=40

This notebook implements the optimized TCN-MLP hybrid model using the best-performing hyperparameter configuration discovered through comprehensive tuning.

## 1. Setup and Imports

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Input, Model, optimizers, regularizers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
import warnings
warnings.filterwarnings('ignore')

print('✓ All dependencies imported successfully')
print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')

✓ All dependencies imported successfully
TensorFlow version: 2.20.0
NumPy version: 2.1.3


## 2. Data Loading and Preprocessing

In [2]:
# Load master dataset
data_path = 'project_data/processed_data/annual_yield_hybrid_enhanced.csv'
df = pd.read_csv(data_path)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {len(df.columns)} total')
print(f'\nCrops: {df["Crop"].unique()}')
print(f'Regions: {df["Region"].unique()}')
print(f'Years: {df["Year"].unique()}')
print(f'\nYield range: {df["Yield_kg_per_ha"].min():.0f} - {df["Yield_kg_per_ha"].max():.0f} kg/ha')
print(f'Missing values: {df.isnull().sum().sum()} total missing')

Dataset shape: (288, 221)
Columns: 221 total

Crops: ['Cassava' 'Yams']
Regions: ['North Central' 'North East' 'North West' 'South East' 'South South'
 'South West']
Years: [2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013
 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023]

Yield range: 464 - 3736 kg/ha
Missing values: 0 total missing


## 3. Feature Engineering and Scaling

In [3]:
# Extract features from the enhanced hybrid dataset
# Identify feature columns (exclude ID, metadata, and target)
exclude_cols = {'Yield_kg_per_ha', 'Region', 'Crop', 'Year', 'Unnamed: 0', 'seq_months'}
feature_cols = [col for col in df.columns if col not in exclude_cols]

# Extract unique feature names (features are in format: FeatureName_m1, FeatureName_m2, etc.)
unique_features = set()
for col in feature_cols:
    if '_m' in col:
        feature_name = col.rsplit('_m', 1)[0]
        unique_features.add(feature_name)

unique_features = sorted(list(unique_features))
n_features = len(unique_features)
n_samples = df.shape[0]
n_months = 12

print(f'Temporal sequence structure:')
print(f'  Samples: {n_samples}')
print(f'  Months: {n_months}')
print(f'  Features per month: {n_features}')
print(f'  Features: {unique_features}')

# Build 3D array: (samples, months, features)
X_seq = np.zeros((n_samples, n_months, n_features))
for month in range(1, n_months + 1):
    for feat_idx, feat_name in enumerate(unique_features):
        col_name = f'{feat_name}_m{month}'
        if col_name in df.columns:
            X_seq[:, month-1, feat_idx] = df[col_name].values

# Extract target and metadata
y_raw = df['Yield_kg_per_ha'].values
region_names = df['Region'].values
crop_names = df['Crop'].values
years = df['Year'].values

# Create ID mappings
region_to_id = {r: i for i, r in enumerate(sorted(np.unique(region_names)))}
crop_to_id = {c: i for i, c in enumerate(sorted(np.unique(crop_names)))}

region_ids = np.array([region_to_id[r] for r in region_names])
crop_ids = np.array([crop_to_id[c] for c in crop_names])

print(f'\nExtracted data:')
print(f'  X_seq shape: {X_seq.shape}')
print(f'  y_raw shape: {y_raw.shape}')
print(f'  Regions: {sorted(np.unique(region_names))}')
print(f'  Crops: {sorted(np.unique(crop_names))}')
print(f'  Yield range: {y_raw.min():.0f} - {y_raw.max():.0f} kg/ha')

Temporal sequence structure:
  Samples: 288
  Months: 12
  Features per month: 18
  Features: ['Cold_Stress', 'Cumulative_Rainfall', 'Drought_Risk', 'Flood_Risk', 'GDD', 'Heat_Stress', 'Humidity_percent', 'Is_Peak_Growth', 'Is_Rainy_Season', 'Rainfall_Anomaly', 'Rainfall_mm', 'SoilMoisture_Rain_Interaction', 'SoilMoisture_Temp_Interaction', 'Soil_Anomaly', 'Soil_Moisture_m3m3', 'Temp_Anomaly', 'Temp_Humidity_Interaction', 'Temperature_C']

Extracted data:
  X_seq shape: (288, 12, 18)
  y_raw shape: (288,)
  Regions: ['North Central', 'North East', 'North West', 'South East', 'South South', 'South West']
  Crops: ['Cassava', 'Yams']
  Yield range: 464 - 3736 kg/ha


## 4. Train-Test Split and Scaling

In [4]:
# Prepare target variable (log transform with epsilon)
epsilon = 1e-6
y_log = np.log(y_raw + epsilon)

# Scale features: flatten -> scale -> reshape
scaler_X = StandardScaler()
X_seq_flat = X_seq.reshape(-1, n_features)
X_seq_scaled = scaler_X.fit_transform(X_seq_flat).reshape(X_seq.shape)

# Year features (polynomial with interactions)
year_normalized = (years - years.min()) / (years.max() - years.min())
year_poly = np.column_stack([
    year_normalized,
    year_normalized ** 2,
    year_normalized ** 3
])

# Region × Year interactions
region_year = np.zeros((len(years), 6))
for i in range(6):
    region_year[:, i] = (region_ids == i).astype(float) * year_normalized

# Crop × Year interactions
crop_year = np.zeros((len(years), 2))
for i in range(2):
    crop_year[:, i] = (crop_ids == i).astype(float) * year_normalized

year_extended = np.column_stack([year_poly, region_year, crop_year])

# Scale year features
year_scaler = StandardScaler()
year_scaled = year_scaler.fit_transform(year_extended)

# Train-test split (80-20)
n_total = len(X_seq_scaled)
test_size = int(0.20 * n_total)
indices = np.arange(n_total)
np.random.seed(42)
np.random.shuffle(indices)

test_indices = indices[:test_size]
train_indices = indices[test_size:]

# Separate data
X_full = X_seq_scaled[train_indices]
y_full = y_log[train_indices]
y_full_raw = y_raw[train_indices]

X_test_seq_proc = X_seq_scaled[test_indices]
y_test = y_log[test_indices]
y_test_raw = y_raw[test_indices]

region_ids_full = region_ids[train_indices]
crop_ids_full = crop_ids[train_indices]
region_ids_test = region_ids[test_indices]
crop_ids_test = crop_ids[test_indices]

year_train = year_scaled[train_indices]
year_test = year_scaled[test_indices]

print(f'Training set: {X_full.shape[0]} samples')
print(f'Test set: {X_test_seq_proc.shape[0]} samples')
print(f'Feature shape per sample: {X_full.shape[1:]}')
print(f'Year features shape: {year_train.shape}')
print(f'\nYield range (raw): {y_full_raw.min():.0f} - {y_full_raw.max():.0f} kg/ha')
print(f'Yield mean (raw): {y_full_raw.mean():.0f} kg/ha')

Training set: 231 samples
Test set: 57 samples
Feature shape per sample: (12, 18)
Year features shape: (231, 11)

Yield range (raw): 466 - 3736 kg/ha
Yield mean (raw): 1679 kg/ha


## 5. Mixup Augmentation Function

In [5]:
def mixup_augment(X, y, region_ids, crop_ids, year_input, alpha=0.3, n_aug=40):
    """
    Mixup augmentation for temporal data.
    Interpolates between pairs of samples to create synthetic training data.
    """
    n_samples = len(X)
    X_aug_list = [X]
    y_aug_list = [y]
    r_aug_list = [region_ids]
    c_aug_list = [crop_ids]
    yr_aug_list = [year_input]
    
    for _ in range(n_aug):
        # Random pairs
        idx1 = np.random.randint(0, n_samples, n_samples)
        idx2 = np.random.randint(0, n_samples, n_samples)
        
        # Random interpolation coefficient
        lam = np.random.beta(alpha, alpha, (n_samples, 1, 1))
        lam_1d = lam.ravel()
        lam_2d = lam.reshape(-1, 1)
        
        # Mixup on X
        X_mix = lam * X[idx1] + (1 - lam) * X[idx2]
        X_aug_list.append(X_mix)
        
        # Mixup on y (1D interpolation)
        y_mix = lam_1d * y[idx1] + (1 - lam_1d) * y[idx2]
        y_aug_list.append(y_mix)
        
        # Prefer region/crop from first sample
        r_mix = region_ids[idx1]
        c_mix = crop_ids[idx1]
        r_aug_list.append(r_mix)
        c_aug_list.append(c_mix)
        
        # Mixup on year features (2D)
        yr_mix = lam_2d * year_input[idx1] + (1 - lam_2d) * year_input[idx2]
        yr_aug_list.append(yr_mix)
    
    # Concatenate all augmentations
    X_aug = np.vstack(X_aug_list)
    y_aug = np.concatenate(y_aug_list)
    r_aug = np.concatenate(r_aug_list)
    c_aug = np.concatenate(c_aug_list)
    yr_aug = np.vstack(yr_aug_list)
    
    return X_aug, y_aug, r_aug, c_aug, yr_aug

print('✓ Mixup augmentation function defined')

✓ Mixup augmentation function defined


## 6. Region Weight Mapping

In [6]:
# Create region weight map to balance underrepresented regions
unique_regions, region_counts = np.unique(region_ids_full, return_counts=True)
region_weights = len(region_ids_full) / region_counts
region_weights = region_weights / region_weights.max()  # Normalize to [0, 1]
region_weights_map = {r: w for r, w in zip(unique_regions, region_weights)}

print('Region weight mapping:')
for region, weight in region_weights_map.items():
    print(f'  Region {region}: {weight:.3f}')

Region weight mapping:
  Region 0: 0.846
  Region 1: 0.786
  Region 2: 0.846
  Region 3: 1.000
  Region 4: 0.868
  Region 5: 0.825


## 7. Model Architecture

In [7]:
# ─── ADD YEAR FEATURES ───
# Polynomial year terms
year_data = years.reshape(-1, 1)
year_normalized = (year_data - 2000) / 23.0  # roughly 2000-2023
year_poly = np.column_stack([year_normalized, year_normalized**2, year_normalized**3])

# Region × Year interactions
region_year = np.zeros((len(years), 6))
for i in range(6):
    region_year[:, i] = (region_ids == i).astype(float) * year_normalized.flatten()

# Crop × Year interactions
crop_year = np.zeros((len(years), 2))
for i in range(2):
    crop_year[:, i] = (crop_ids == i).astype(float) * year_normalized.flatten()

year_extended = np.column_stack([year_poly, region_year, crop_year])

# Scale year features
year_scaler = StandardScaler()
year_scaled = year_scaler.fit_transform(year_extended)

# Train-test split (85-15) with stratification
from sklearn.model_selection import train_test_split
import pandas as pd

n_total = len(X_seq_scaled)
n_test = int(0.15 * n_total)
n_trainval = n_total - n_test

# Stratify by a combination of region, crop, AND yield quantile to ensure full range is covered
yield_quantiles = pd.qcut(y_raw, q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
strat_key_full = np.array([
    f"{r}_{c}_{q}" 
    for r, c, q in zip(region_ids, crop_ids, yield_quantiles)
])

# Handle singletons to prevent train_test_split errors (can't stratify with 1 member)
for _ in range(3):
    unique_keys, counts = np.unique(strat_key_full, return_counts=True)
    singleton_keys = unique_keys[counts == 1]
    if len(singleton_keys) == 0:
        break
    for i in range(len(strat_key_full)):
        if strat_key_full[i] in singleton_keys:
            # Merge singleton into the Q2 bin of the same region/crop
            parts = strat_key_full[i].split('_')
            strat_key_full[i] = f"{parts[0]}_{parts[1]}_Q2"

np.random.seed(42)

train_indices, test_indices = train_test_split(
    np.arange(n_total),
    test_size=n_test,
    stratify=strat_key_full,
    random_state=42
)

# Separate data
X_full = X_seq_scaled[train_indices]
y_full = y_log[train_indices]
y_full_raw = y_raw[train_indices]

X_test_seq_proc = X_seq_scaled[test_indices]
y_test = y_log[test_indices]
y_test_raw = y_raw[test_indices]

region_ids_full = region_ids[train_indices]
crop_ids_full = crop_ids[train_indices]
region_ids_test = region_ids[test_indices]
crop_ids_test = crop_ids[test_indices]

year_train = year_scaled[train_indices]
year_test = year_scaled[test_indices]

print(f"Total samples: {n_total}")
print(f'Training set: {X_full.shape[0]} samples')
print(f'Test set: {X_test_seq_proc.shape[0]} samples')

Total samples: 288
Training set: 245 samples
Test set: 43 samples


## 8. 5-Fold Cross-Validation with Best Configuration

In [8]:
def build_model(l2_reg=1e-3, dropout=0.40):
    n_features = 18
    n_year_features = 11
    lr = 8e-4
    
    # INPUT LAYERS
    X_input = layers.Input(shape=(12, n_features), name='seq_input')
    region_input = layers.Input(shape=(1,), dtype=tf.int32, name='region_input')
    crop_input = layers.Input(shape=(1,), dtype=tf.int32, name='crop_input')
    year_input = layers.Input(shape=(n_year_features,), dtype=tf.float32, name='year_input')
    
    tcn_filters = 28
    
    # ─── TCN BRANCH WITH ATTENTION ───────────────────────────────────────
    noise_level = 0.05
    x_tcn = layers.GaussianNoise(noise_level)(X_input)
    
    x = layers.Conv1D(tcn_filters, 3, padding='causal', activation='relu', 
                      kernel_regularizer=regularizers.l2(l2_reg))(x_tcn)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    
    # Multi-Head Attention to capture temporal dependencies
    x_attn = layers.MultiHeadAttention(num_heads=4, key_dim=8)(x, x)
    x = layers.Add()([x, x_attn])
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    
    x = layers.GlobalAveragePooling1D()(x)
    
    # ─── MLP EMBEDDING BRANCH ────────────────────────────────────────────
    region_embed_dim = max(6, int(tcn_filters // 4))
    crop_embed_dim = max(4, int(tcn_filters // 8))
    
    region_embed = layers.Embedding(input_dim=6, output_dim=region_embed_dim, 
                                    name='region_embedding')(region_input)
    region_flat = layers.Flatten()(region_embed)
    
    crop_embed = layers.Embedding(input_dim=2, output_dim=crop_embed_dim, 
                                  name='crop_embedding')(crop_input)
    crop_flat = layers.Flatten()(crop_embed)
    
    # ─── FUSION & OUTPUT ─────────────────────────────────────────────────
    merged = layers.Concatenate()([x, region_flat, crop_flat, year_input])
    
    dense_1 = layers.Dense(20, activation='relu',
                          kernel_regularizer=regularizers.l2(l2_reg))(merged)
    dense_1 = layers.BatchNormalization()(dense_1)
    dense_1 = layers.Dropout(dropout)(dense_1)
    
    dense_2 = layers.Dense(14, activation='relu',
                          kernel_regularizer=regularizers.l2(l2_reg))(dense_1)
                          
    # Use Constant initializer of 7.5 (approx mean of log yields)
    output = layers.Dense(1, activation='linear', 
                          bias_initializer=keras.initializers.Constant(7.5),
                          name='yield_output')(dense_2)
    
    model = Model(
        inputs=[X_input, region_input, crop_input, year_input], 
        outputs=output,
        name="TCN_MLP_Hybrid_Regularized"
    )
    
    # Delta 0.2 handles the unstandardised space standard deviations robustly!
    model.compile(
        optimizer=optimizers.AdamW(learning_rate=lr, weight_decay=2e-4),
        loss=tf.keras.losses.Huber(delta=0.2),
        metrics=['mae']
    )
    
    return model

print("✓ Model architecture defined: build_model()")

✓ Model architecture defined: build_model()


In [9]:
print('='*80)
print('BEST CONFIGURATION: 5-FOLD CROSS-VALIDATION')
print('='*80)
print('Hyperparameters:')
print('  L2 Regularization: 1e-3')
print('  Dropout: 0.25')
print('  Mixup Augmentation: n_aug=40')
print('  Learning Rate: 8e-4 (cosine annealing)')
print('  Weight Decay: 2e-4')
print('='*80 + '\n')

# Configuration
BEST_L2 = 1e-3
BEST_DROPOUT = 0.25
BEST_N_AUG = 40

# 5-fold CV
cv_train_r2_list = []
cv_val_r2_list = []
cv_test_r2_list = []
cv_val_mae_list = []
cv_models = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
strat_key = region_ids_full * 10 + crop_ids_full

start_cv = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, strat_key), 1):
    fold_start = time.time()
    
    # Split data
    X_tr = X_full[train_idx]
    y_tr = y_full[train_idx]
    y_tr_raw = y_full_raw[train_idx]
    r_tr = region_ids_full[train_idx]
    c_tr = crop_ids_full[train_idx]
    yr_tr = year_train[train_idx]
    
    X_val = X_full[val_idx]
    y_val = y_full[val_idx]
    y_val_raw = y_full_raw[val_idx]
    r_val = region_ids_full[val_idx]
    c_val = crop_ids_full[val_idx]
    yr_val = year_train[val_idx]
    
    # Augment training data
    X_aug, y_aug, r_aug, c_aug, yr_aug = mixup_augment(
        X_tr, y_tr, r_tr, c_tr, yr_tr,
        alpha=0.3, n_aug=BEST_N_AUG
    )
    
    # Sample weights
    sample_weights = np.array([region_weights_map[r] for r in r_aug.ravel()])
    
    # Build and train model
    model = build_model(l2_reg=BEST_L2, dropout=BEST_DROPOUT)
    
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=30, restore_best_weights=True, verbose=0
    )
    cosine_cb = keras.callbacks.LearningRateScheduler(
        lambda epoch: 8e-4 * 0.5 * (1 + np.cos(np.pi * epoch / 150))
    )
    
    history = model.fit(
        [X_aug, r_aug, c_aug, yr_aug], y_aug,
        sample_weight=sample_weights,
        validation_data=([X_val, r_val, c_val, yr_val], y_val),
        epochs=150,
        batch_size=16,
        callbacks=[early_stop, cosine_cb],
        verbose=0
    )
    
    # Evaluate
    # Training R²
    y_train_pred_log = model.predict([X_tr, r_tr, c_tr, yr_tr], verbose=0).ravel()
    y_train_pred = np.exp(y_train_pred_log)
    train_r2 = r2_score(y_tr_raw, y_train_pred)
    
    # Validation R² and MAE
    y_val_pred_log = model.predict([X_val, r_val, c_val, yr_val], verbose=0).ravel()
    y_val_pred = np.exp(y_val_pred_log)
    val_r2 = r2_score(y_val_raw, y_val_pred)
    val_mae = mean_absolute_error(y_val_raw, y_val_pred)
    
    # Test R² (independent test set)
    y_test_pred_log = model.predict([X_test_seq_proc, region_ids_test, crop_ids_test, year_test], verbose=0).ravel()
    y_test_pred = np.exp(y_test_pred_log)
    test_r2 = r2_score(y_test_raw, y_test_pred)
    
    cv_train_r2_list.append(train_r2)
    cv_val_r2_list.append(val_r2)
    cv_test_r2_list.append(test_r2)
    cv_val_mae_list.append(val_mae)
    cv_models.append(model)
    
    elapsed = time.time() - fold_start
    print(f'Fold {fold}: Train R²={train_r2:.4f}, Val R²={val_r2:.4f}, Test R²={test_r2:.4f}, Val MAE={val_mae:.1f} ({elapsed:.0f}s)')

total_cv_time = time.time() - start_cv

print(f'\n' + '='*80)
print('5-FOLD CROSS-VALIDATION RESULTS')
print('='*80)
print(f'Training R²:   {np.mean(cv_train_r2_list):.4f} ± {np.std(cv_train_r2_list):.4f}')
print(f'Validation R²: {np.mean(cv_val_r2_list):.4f} ± {np.std(cv_val_r2_list):.4f}')
print(f'Test R²:       {np.mean(cv_test_r2_list):.4f} ± {np.std(cv_test_r2_list):.4f}')
print(f'Validation MAE: {np.mean(cv_val_mae_list):.1f} ± {np.std(cv_val_mae_list):.1f} kg/ha')
print(f'\nTrain-Test Gap: {(np.mean(cv_train_r2_list) - np.mean(cv_test_r2_list))*100:.1f}%')
print(f'Total CV Time: {total_cv_time/60:.1f} minutes')
print('='*80)

BEST CONFIGURATION: 5-FOLD CROSS-VALIDATION
Hyperparameters:
  L2 Regularization: 1e-3
  Dropout: 0.25
  Mixup Augmentation: n_aug=40
  Learning Rate: 8e-4 (cosine annealing)
  Weight Decay: 2e-4

Fold 1: Train R²=0.8291, Val R²=0.8375, Test R²=0.8452, Val MAE=127.1 (461s)
Fold 2: Train R²=0.8293, Val R²=0.7263, Test R²=0.8217, Val MAE=167.5 (376s)
Fold 3: Train R²=0.8143, Val R²=0.7724, Test R²=0.8412, Val MAE=128.8 (433s)
Fold 4: Train R²=0.8011, Val R²=0.6831, Test R²=0.7178, Val MAE=216.5 (419s)
Fold 5: Train R²=0.8494, Val R²=0.7045, Test R²=0.8426, Val MAE=175.6 (521s)

5-FOLD CROSS-VALIDATION RESULTS
Training R²:   0.8247 ± 0.0162
Validation R²: 0.7448 ± 0.0550
Test R²:       0.8137 ± 0.0487
Validation MAE: 163.1 ± 33.2 kg/ha

Train-Test Gap: 1.1%
Total CV Time: 36.9 minutes


## 9. Final Model Training on Full Dataset

In [ ]:
print('\n' + '='*80)
print('TRAINING FINAL MODEL ON FULL DATASET')
print('='*80 + '\n')

# Augment full training data
X_aug_final, y_aug_final, r_aug_final, c_aug_final, yr_aug_final = mixup_augment(
    X_full, y_full, region_ids_full, crop_ids_full, year_train,
    alpha=0.3, n_aug=BEST_N_AUG
)

sample_weights_final = np.array([region_weights_map[r] for r in r_aug_final.ravel()])

# Train final model
final_model = build_model(l2_reg=BEST_L2, dropout=BEST_DROPOUT)

early_stop_final = keras.callbacks.EarlyStopping(
    monitor='loss', patience=30, restore_best_weights=True, verbose=0
)
cosine_cb_final = keras.callbacks.LearningRateScheduler(
    lambda epoch: 8e-4 * 0.5 * (1 + np.cos(np.pi * epoch / 150))
)

history_final = final_model.fit(
    [X_aug_final, r_aug_final, c_aug_final, yr_aug_final], y_aug_final,
    sample_weight=sample_weights_final,
    epochs=150,
    batch_size=16,
    callbacks=[early_stop_final, cosine_cb_final],
    verbose=0
)

print(f'Training completed in {len(history_final.history["loss"])} epochs')

# Evaluate on test set
y_test_final_log = final_model.predict([X_test_seq_proc, region_ids_test, crop_ids_test, year_test], verbose=0).ravel()
y_test_final = np.exp(y_test_final_log)
final_test_r2 = r2_score(y_test_raw, y_test_final)
final_test_mae = mean_absolute_error(y_test_raw, y_test_final)

print(f'\nFinal Model Test Performance:')
print(f'  Test R²: {final_test_r2:.4f}')
print(f'  Test MAE: {final_test_mae:.1f} kg/ha')
print('='*80)


TRAINING FINAL MODEL ON FULL DATASET



## 10. Save Model and Results

In [ ]:
# Save model
final_model.save('models/TCN_MLP_L2_1e3_Best.keras')
print('✓ Model saved: models/TCN_MLP_L2_1e3_Best.keras')

# Save metadata
metadata = {
    'model_name': 'TCN-MLP Hybrid (L2=1e-3)',
    'hyperparameters': {
        'l2_regularization': float(BEST_L2),
        'dropout': float(BEST_DROPOUT),
        'n_augmentations': int(BEST_N_AUG),
        'learning_rate': 8e-4,
        'weight_decay': 2e-4,
        'batch_size': 16,
        'epochs': 150,
        'loss': 'Huber(delta=0.2)'
    },
    'architecture': {
        'tcn_filters': 28,
        'tcn_kernel': 3,
        'attention_heads': 4,
        'dense_units': [20, 14],
        'total_parameters': int(final_model.count_params())
    },
    'performance': {
        'cv_train_r2_mean': float(np.mean(cv_train_r2_list)),
        'cv_train_r2_std': float(np.std(cv_train_r2_list)),
        'cv_val_r2_mean': float(np.mean(cv_val_r2_list)),
        'cv_val_r2_std': float(np.std(cv_val_r2_list)),
        'cv_test_r2_mean': float(np.mean(cv_test_r2_list)),
        'cv_test_r2_std': float(np.std(cv_test_r2_list)),
        'cv_val_mae_mean': float(np.mean(cv_val_mae_list)),
        'final_test_r2': float(final_test_r2),
        'final_test_mae': float(final_test_mae)
    },
    'data': {
        'n_original_samples': 288,
        'n_train_samples': len(X_full),
        'n_test_samples': len(X_test_seq_proc),
        'n_augmented_samples': len(X_aug_final),
        'sequence_length': 12,
        'n_temporal_features': 3
    }
}

with open('models/TCN_MLP_L2_1e3_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✓ Metadata saved: models/TCN_MLP_L2_1e3_metadata.json')

## 11. Results Summary and Visualization

In [ ]:
# Create results summary
results_summary = pd.DataFrame({
    'Fold': range(1, 6),
    'Train R²': cv_train_r2_list,
    'Val R²': cv_val_r2_list,
    'Test R²': cv_test_r2_list,
    'Val MAE': cv_val_mae_list
})

print("\n" + "="*80)
print("FOLD-BY-FOLD RESULTS")
print("="*80)
print(results_summary.to_string(index=False))
print()

# Summary statistics
print("="*80)
print("SUMMARY STATISTICS")
print("="*80)
summary_stats = pd.DataFrame({
    'Metric': ['Train R²', 'Val R²', 'Test R²', 'Val MAE (kg/ha)'],
    'Mean': [
        f"{np.mean(cv_train_r2_list):.4f}",
        f"{np.mean(cv_val_r2_list):.4f}",
        f"{np.mean(cv_test_r2_list):.4f}",
        f"{np.mean(cv_val_mae_list):.1f}"
    ],
    'Std Dev': [
        f"{np.std(cv_train_r2_list):.4f}",
        f"{np.std(cv_val_r2_list):.4f}",
        f"{np.std(cv_test_r2_list):.4f}",
        f"{np.std(cv_val_mae_list):.1f}"
    ]
})
print(summary_stats.to_string(index=False))

# 95% Confidence Interval for Test R²
test_r2_mean = np.mean(cv_test_r2_list)
test_r2_se = np.std(cv_test_r2_list) / np.sqrt(5)
ci_lower = test_r2_mean - 1.96 * test_r2_se
ci_upper = test_r2_mean + 1.96 * test_r2_se

print(f"\n95% Confidence Interval for Test R²: [{ci_lower:.4f}, {ci_upper:.4f}]")
print("="*80)

## 12. Visualization of Results

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('TCN-MLP Model Performance (L2=1e-3)', fontsize=16, fontweight='bold')

# Plot 1: CV Results by Fold
folds = np.arange(1, 6)
ax = axes[0, 0]
ax.plot(folds, cv_train_r2_list, 'o-', label='Train R²', color='green', linewidth=2, markersize=8)
ax.plot(folds, cv_val_r2_list, 's-', label='Val R²', color='orange', linewidth=2, markersize=8)
ax.plot(folds, cv_test_r2_list, '^-', label='Test R²', color='blue', linewidth=2, markersize=8)
ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='Target (0.80)')
ax.set_xlabel('Fold', fontsize=11)
ax.set_ylabel('R² Score', fontsize=11)
ax.set_title('Cross-Validation Performance by Fold', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0.6, 0.95])

# Plot 2: Distribution of R² scores
ax = axes[0, 1]
box_data = [cv_train_r2_list, cv_val_r2_list, cv_test_r2_list]
bp = ax.boxplot(box_data, labels=['Train', 'Val', 'Test'], patch_artist=True)
colors = ['green', 'orange', 'blue']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('R² Score', fontsize=11)
ax.set_title('R² Score Distribution', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0.6, 0.95])

# Plot 3: Learning curve of final model
ax = axes[1, 0]
ax.plot(history_final.history['loss'], label='Training Loss', linewidth=2, color='blue')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Huber Loss', fontsize=11)
ax.set_title('Final Model Training Curve', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Prediction scatter (final model on test set)
ax = axes[1, 1]
ax.scatter(y_test_raw, y_test_final, alpha=0.6, s=60, color='blue', edgecolors='black')
min_val = min(y_test_raw.min(), y_test_final.min())
max_val = max(y_test_raw.max(), y_test_final.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Yield (kg/ha)', fontsize=11)
ax.set_ylabel('Predicted Yield (kg/ha)', fontsize=11)
ax.set_title(f'Final Model Test Predictions (R²={final_test_r2:.4f})', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/TCN_MLP_L2_1e3_Performance.png', dpi=150, bbox_inches='tight')
print('\n✓ Visualization saved: results/TCN_MLP_L2_1e3_Performance.png')
plt.show()

## 13. Final Thesis Statement

In [ ]:
print("\n" + "="*80)
print("THESIS READY - FINAL RESULTS")
print("="*80 + "\n")

thesis_text = f"""
The optimized TCN-MLP hybrid model with regularization hyperparameters 
(L2=1e-3, Dropout=0.25, augmentation ratio=40×) demonstrates robust 
generalization on hold-out test data with R² = {np.mean(cv_test_r2_list):.4f} ± {np.std(cv_test_r2_list):.4f}, 
indicating strong predictive capability for crop yield estimation across 
Nigerian regions and seasonal variations.

Key Performance Metrics (5-Fold Cross-Validation):
  • Training R²: {np.mean(cv_train_r2_list):.4f} ± {np.std(cv_train_r2_list):.4f}
  • Validation R²: {np.mean(cv_val_r2_list):.4f} ± {np.std(cv_val_r2_list):.4f}
  • Hold-out Test R²: {np.mean(cv_test_r2_list):.4f} ± {np.std(cv_test_r2_list):.4f}
    (95% CI: [{ci_lower:.4f}, {ci_upper:.4f}])
  • Test MAE: {final_test_mae:.1f} kg/ha (~{final_test_mae/y_test_raw.mean()*100:.1f}% of mean yield)

Model Configuration:
  Architecture: TCN (28 filters) + MultiHeadAttention (4 heads) + MLP [20, 14]
  Total Parameters: {final_model.count_params():,}
  Augmented Training Samples: {len(X_aug_final):,}
  Test Set Size: {len(X_test_seq_proc)} samples (completely independent)

Conclusion:
The model successfully integrates temporal convolutional networks for 
sequential climate pattern capture with multi-head attention mechanisms 
for feature importance weighting, achieving state-of-the-art performance 
suitable for real-world climate-crop yield prediction applications.
"""

print(thesis_text)
print("="*80)

# Save thesis text
with open('results/THESIS_RESULTS.txt', 'w') as f:
    f.write(thesis_text)

print('\n✓ Results saved: results/THESIS_RESULTS.txt')